## IMPORT LIBRARY

In [8]:
%pip install folium


Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import json
import re
from folium import Element

## IMPORT DATA

In [13]:
# 1. Baca data CSV
csv_file = "data_tpt_jateng.csv"
df = pd.read_csv(csv_file, on_bad_lines='skip') 

kolom_wilayah = df.columns[0]
kolom_tpt = df.columns[1]
daftar_wilayah_csv = df[kolom_wilayah].dropna().unique().tolist()

# 2. Baca file GeoJSON
geojson_file = "peta_jateng.geojson"
with open(geojson_file, 'r', encoding='utf-8') as f:
    geojson_data = json.load(f)

# Filter fitur GeoJSON yang valid
geojson_data['features'] = [
    f for f in geojson_data['features'] 
    if f.get('properties') is not None and f['properties'].get('WADMKK') is not None
]

## CLEANING DATA

In [14]:
# 3. Fungsi pencocokan cerdas nama wilayah
def cocokkan_nama(nama_geojson, daftar_csv):
    nama_bersih = str(nama_geojson).strip().upper()
    for nama_csv in daftar_csv:
        if nama_bersih == str(nama_csv).strip().upper():
            return nama_csv
            
    nama_kunci_geo = re.sub(r'^(KABUPATEN|KOTA)\s+', '', nama_bersih)
    for nama_csv in daftar_csv:
        nama_kunci_csv = re.sub(r'^(KABUPATEN|KOTA)\s+', '', str(nama_csv).strip().upper())
        if nama_kunci_geo == nama_kunci_csv:
            is_kota_geo = 'KOTA' in nama_bersih
            is_kota_csv = 'KOTA' in str(nama_csv).strip().upper()
            if is_kota_geo == is_kota_csv:
                return nama_csv
    return nama_geojson

# 4. Sinkronisasi nama dan nilai TPT
for feature in geojson_data['features']:
    nama_lama = feature['properties']['WADMKK']
    nama_baru = cocokkan_nama(nama_lama, daftar_wilayah_csv)
    
    feature['properties']['WADMKK'] = nama_baru
    
    baris_data = df[df[kolom_wilayah] == nama_baru]
    if not baris_data.empty:
        feature['properties']['TPT'] = float(baris_data.iloc[0][kolom_tpt])
    else:
        feature['properties']['TPT'] = None

## MEMBUAT PETA JAWA TENGAH

In [18]:
lat_jateng = -7.150975
lon_jateng = 110.140259
zoom_jateng = 8

peta = folium.Map(location=[lat_jateng, lon_jateng], zoom_start=zoom_jateng, tiles=None)

folium.TileLayer('OpenStreetMap', name='Peta Standar').add_to(peta)

folium.Choropleth(
    geo_data=geojson_data,
    name='Choropleth TPT',
    data=df,
    columns=[kolom_wilayah, kolom_tpt],
    key_on='feature.properties.WADMKK',
    fill_color='YlOrRd', 
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Tingkat Pengangguran Terbuka (%) - Sumber: BPS',
    reset=True
).add_to(peta)

folium.GeoJson(
    geojson_data,
    name="Interaksi Wilayah",
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent'},
    highlight_function=lambda x: {'weight': 3, 'color': 'black', 'fillOpacity': 0.1},
    tooltip=folium.features.GeoJsonTooltip(
        fields=['WADMKK', 'TPT'],
        aliases=['Wilayah:', 'TPT (%):'],
        localize=True,
        sticky=False
    ),
    zoom_on_click=True
).add_to(peta)

# ==========================================
# 3. SCRIPT JAVASCRIPT AMAN (MENGGUNAKAN .format())
# ==========================================
js_template = """
    <script>
    document.addEventListener("DOMContentLoaded", function() {{
        var mapVar = Object.keys(window).find(key => window[key] instanceof L.Map);
        if (mapVar) {{
            var map = window[mapVar];
            map.on('dblclick', function(e) {{
                map.setView([{lat}, {lon}], {zoom});
            }});
        }}
    }});
    </script>
"""

# Masukkan koordinat Jawa Tengah dengan aman menggunakan .format()
macro_element = Element(js_template.format(lat=lat_jateng, lon=lon_jateng, zoom=zoom_jateng))
peta.get_root().html.add_child(macro_element)

# Kotak Keterangan Informasi
title_html = '''
     <div style="position: fixed; 
                 bottom: 50px; left: 50px; width: 350px; height: 125px; 
                 background-color: white; z-index:9999; font-size:13px;
                 border:2px solid grey; border-radius: 5px; padding: 10px;">
     <b>Peta Tingkat Pengangguran Terbuka (TPT) Jateng</b><br>
     - <b>Info:</b> Klik 1x (Zoom In), <b>Double Klik</b> (Reset ke Seluruh Jateng).<br>
     - <b>Sumber Data:</b> Badan Pusat Statistik (BPS)<br>
     - <b>Sumber Peta:</b> GitHub ardian28 & Peta Administrasi
     </div>
'''
peta.get_root().html.add_child(Element(title_html))

folium.LayerControl().add_to(peta)

output_html = "index.html"
peta.save(output_html)
print(f"Selesai! Buka file '{output_html}' di browser Anda.")

Selesai! Buka file 'index.html' di browser Anda.
